# Tokenization and Embeddings

**Goal:** Understand how text becomes numbers — the two-step process that powers all of NLP.

**Prereqs:** Python basics. Familiarity with lists and numpy arrays.

**Libraries:** `transformers`, `sentence-transformers`

## Why Text Needs to Become Numbers

**Go/TS comparison:** In Go, you work with `[]byte` or `string` — text is just data you parse with string functions. In NLP, models can't see text at all. They only see numbers. Tokenization converts text → integer IDs, and embeddings convert those IDs → dense float vectors that capture meaning. Every NLP pipeline starts here.

## Part 1: Tokenization

### Word Tokenization — The Naive Approach

In [ ]:
# The simplest tokenizer: split on spaces
sentence = "The cat sat on the mat."
tokens = sentence.split()
print(f"Tokens: {tokens}")
print(f"Count: {len(tokens)}")

# Problem: punctuation sticks to words
# Problem: "don't" → should it be "do" + "n't"?
sentence2 = "I don't think it's working!"
print(f"\nNaive split: {sentence2.split()}")

**Go/TS comparison:** `strings.Split(s, " ")` in Go or `s.split(' ')` in TS gives you the same naive result. Real tokenization is much harder — punctuation, contractions, unicode, and languages without spaces (Chinese, Japanese) all break simple splitting.

In [ ]:
# Experiment: try to handle punctuation yourself — see why this is hard
import re

def better_tokenize(text):
    """Split on spaces and punctuation, but keep contractions together"""
    return re.findall(r"\w+(?:'\w+)?|[^\w\s]", text)

print(better_tokenize("I don't think it's working!"))
print(better_tokenize("The U.K. startup raised $1.5B"))
# Still breaks on edge cases — this is why we use pretrained tokenizers

### ✍️ In Your Own Words

Why is splitting on spaces insufficient for tokenization? What edge cases break it? Write your answer here.

### Subword Tokenization — What Modern Models Use

**Go/TS comparison:** No equivalent concept. Subword tokenization (BPE, WordPiece) is unique to NLP. It keeps common words whole but splits rare words into known pieces. "unfrigginbelievable" becomes multiple subword tokens. This is the sweet spot between word-level (can't handle unknown words) and character-level (sequences get too long).

In [ ]:
from transformers import AutoTokenizer

# Load BERT's tokenizer — uses WordPiece subword tokenization
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sentence = "The cat sat on the mat"

# Three levels of tokenization:
tokens = tokenizer.tokenize(sentence)        # Human-readable tokens
ids = tokenizer.encode(sentence)             # Integer IDs (what the model sees)
decoded = tokenizer.decode(ids)              # Back to text

print(f"Tokens:  {tokens}")
print(f"IDs:     {ids}")
print(f"Decoded: {decoded}")

In [ ]:
# See how subword tokenization handles unknown words
weird_words = [
    "unfrigginbelievable",
    "supercalifragilistic",
    "ChatGPT",
    "kubernetes",
]

for word in weird_words:
    tokens = tokenizer.tokenize(word)
    print(f"{word:30s} → {tokens}")

In [ ]:
# Experiment: compare BERT vs GPT-2 tokenizers — different strategies
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")

test = "The quick brown fox jumps over the lazy dog"
bert_tokens = tokenizer.tokenize(test)
gpt2_tokens = gpt2_tokenizer.tokenize(test)

print(f"BERT tokens ({len(bert_tokens)}): {bert_tokens}")
print(f"GPT2 tokens ({len(gpt2_tokens)}): {gpt2_tokens}")
print(f"\nBERT vocab size: {tokenizer.vocab_size}")
print(f"GPT2 vocab size: {gpt2_tokenizer.vocab_size}")

### ✍️ In Your Own Words

Why do BERT and GPT-2 tokenize the same text differently? What determines a model's tokenization strategy? Write your answer here.

### Special Tokens

In [ ]:
# Special tokens — metadata the model needs
print(f"Special tokens: {tokenizer.special_tokens_map}")

# See them in action
sentence = "Hello world"
ids = tokenizer.encode(sentence)
tokens = tokenizer.convert_ids_to_tokens(ids)
print(f"\nTokens with special: {tokens}")
print(f"IDs:                 {ids}")

# [CLS] = "start of input" (used for classification tasks)
# [SEP] = "separator" (marks end of input or between sentence pairs)
# [PAD] = "padding" (fills sequences to same length in a batch)

In [ ]:
# Experiment: token count matters for LLM context windows
long_text = "This is a test. " * 100
ids = tokenizer.encode(long_text)
print(f"Characters: {len(long_text)}")
print(f"Words:      {len(long_text.split())}")
print(f"Tokens:     {len(ids)}")
print(f"\nTokens ≠ words. This is why LLM context limits are in tokens, not words.")

### ✍️ In Your Own Words

What are special tokens and why do models need them? Write your answer here.

## Part 2: Embeddings

### From Tokens to Meaning

**Go/TS comparison:** In Go, if you wanted to compare two strings for similarity, you'd use Levenshtein distance or exact matching. That only catches lexical similarity ("cat" vs "bat"). Embeddings capture *semantic* similarity — "cat" and "feline" end up close together in vector space even though they share zero characters.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a sentence embedding model — small and fast
model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed a single sentence
sentence = "The cat sat on the mat"
embedding = model.encode(sentence)

print(f"Input:      '{sentence}'")
print(f"Output:     float array of shape {embedding.shape}")
print(f"Dimensions: {len(embedding)}")
print(f"First 10:   {embedding[:10].round(4)}")
print(f"dtype:      {embedding.dtype}")

In [ ]:
# Embeddings capture meaning, not just words
sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",           # Same meaning, different words
    "The dog chased the ball",               # Different meaning, some shared words
    "bank of the river",                     # Ambiguous word: bank
    "bank account balance",                  # Different meaning of "bank"
]

embeddings = model.encode(sentences)
print(f"Batch shape: {embeddings.shape}")
print(f"\nEach sentence → a {embeddings.shape[1]}-dimensional vector")
print(f"Regardless of sentence length!")

In [ ]:
# Experiment: embedding properties
import numpy as np

# Same input always gives same output (deterministic)
e1 = model.encode("hello world")
e2 = model.encode("hello world")
print(f"Deterministic? {np.allclose(e1, e2)}")

# Empty and very long inputs both work
e_empty = model.encode("")
e_long = model.encode("word " * 500)
print(f"Empty string shape: {e_empty.shape}")
print(f"500-word shape:     {e_long.shape}")
print(f"Same dimensions regardless of input length!")

### ✍️ In Your Own Words

How is semantic similarity (embeddings) different from lexical similarity (string matching)? Why does this matter for RAG? Write your answer here.

### Comparing Embedding Models

In [ ]:
# Different models produce different-sized embeddings
model_small = SentenceTransformer("all-MiniLM-L6-v2")
model_large = SentenceTransformer("all-mpnet-base-v2")

sentence = "The quick brown fox"
e_small = model_small.encode(sentence)
e_large = model_large.encode(sentence)

print(f"MiniLM (small): {e_small.shape[0]} dimensions")
print(f"MPNet (large):  {e_large.shape[0]} dimensions")
print(f"\nMore dimensions = more nuance, but slower and more memory")
print(f"MiniLM is usually good enough for most use cases")

### ✍️ In Your Own Words

Why might you choose a smaller embedding model over a larger one? Write your answer here.

## Recap

- ✅ Tokenization converts text → integer IDs that models can process
- ✅ Subword tokenization (BPE/WordPiece) handles unknown words by splitting into known pieces
- ✅ Different models use different tokenizers and vocabularies
- ✅ Special tokens (`[CLS]`, `[SEP]`, `[PAD]`) provide metadata to models
- ✅ Token count ≠ word count — matters for context windows
- ✅ Embeddings convert text → dense float vectors capturing semantic meaning
- ✅ Similar meanings → nearby vectors, even with different words
- ✅ Embedding dimensions vary by model — more dimensions = more nuance

**Next:** [01_similarity_and_search.ipynb](./01_similarity_and_search.ipynb) — use these embeddings to find similar text